In [ ]:
### This noteboook is used for dealing with problem in ContiguousSNP
import sys
import os
import zipfile
from scipy import stats

## import function from python_scripts
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
from python_scripts.gene_level_MoM import * 
from python_scripts.pairwise_product_var_calculation import * 
from python_scripts.visualization import * 
from python_scripts.weight_function import * 
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.linalg import cholesky
import time


In [ ]:
###unzip file
zip_files = [
    "../test_data/chr1_Contiguous1KSNP.zip",
    "../test_data/chr1_Random1KSNP.zip"
]
for zf in zip_files:
    with zipfile.ZipFile(zf, "r") as z:
        z.extractall("test_data")   
    print(f"Unzipped: {zf}")
##read data  
##Random snp
RandomSNP = pd.read_csv("test_data/chr1_Random1KSNP.raw", sep=r'\s+')
RandomSNP = RandomSNP.iloc[:, 6:].to_numpy()
##Contiguous snp
ContiguousSNP = pd.read_csv("test_data/chr1_Contiguous1KSNP.raw", sep=r'\s+')
ContiguousSNP = ContiguousSNP.iloc[:, 6:].to_numpy()

In [178]:

###approximate mom
def simulate_Cholesky_from_std(real_data, s2a=0.9, s2e=0.1, stability=1e-10):

    Z = (real_data - real_data.mean(axis=0)) / real_data.std(axis=0)
    n, m = Z.shape

    K = (Z @ Z.T) / m
    Sigma_a = s2a * K + stability * np.eye(n)
    La = cholesky(Sigma_a, lower=True) 
    return La


def simulate_from_raw_additive_remove_sampling_err(real_data, La, s2a=0.9, s2e=0.1):

    Z = (real_data - real_data.mean(axis=0)) / real_data.std(axis=0)
    n, m = Z.shape

    u1 = np.random.randn(n)
    u2 = np.random.randn(n)

    a = La @ u1
    Le = np.sqrt(s2e) * np.eye(n)
    e = Le @ u2

    cur_var_a = np.var(a, ddof=0)
    scale_a = np.sqrt(s2a / cur_var_a)
    a = a * scale_a
    

    cur_var_e = np.var(e, ddof=0)
    scale_e = np.sqrt(s2e / cur_var_e)
    e = e * scale_e
    

    y = a + e
    y -= y.mean()

    return Z, y


def MoM_additive(Z, y, nmc=100):

    n = Z.shape[0]
    m = Z.shape[1]
    u = np.random.choice([-1, 1], size=(n, nmc))

    Ku = (Z @ (Z.T @ u)) / m
    # tr(K)
    trK_app = np.mean(np.sum(u * Ku, axis=0))

    # tr(K^2)
    trK2_app = np.mean(np.sum(Ku * Ku, axis=0))

    # quadratic forms
    Zty = Z.T @ y
    yKy = (Zty @ Zty) / m        # y^T K y = (1/m) ||Z^T y||^2
    yy  = y @ y                  # y^T y

    # MoM linear system: A @ [s2a, s2e].T = b
    A = np.array([[trK2_app, trK_app],
                  [trK_app,  n      ]])
    b = np.array([yKy, yy])

    # Solve A @ [s2a, s2e].T = b
    x = np.linalg.solve(A, b)
    return x[0], x[1], A


def MoM_additive_exact(Z, y):

    n, m = Z.shape

    # tr(K) = (1/m) * sum_j ||Z_j||^2
    # Equals n exactly when Z is column-standardized with ddof=0.
    trK = np.sum(Z * Z) / m

    # tr(K^2) = (1/m^2) * ||Z^T Z||_F^2
    ZtZ = Z.T @ Z
    trK2 = np.sum(ZtZ * ZtZ) / (m * m)

    # y^T K y = (1/m) * ||Z^T y||^2
    Zty = Z.T @ y
    yKy = (Zty @ Zty) / m

    # y^T y
    yy = y @ y

    # MoM 2x2 linear system: A @ [s2a, s2e].T = b
    A = np.array([[trK2, trK],
                  [trK,  n  ]])
    b = np.array([yKy, yy])

    x = np.linalg.solve(A, b)
    return x[0], x[1], A
    

In [180]:
#X =RandomSNP[:1000,]
X = ContiguousSNP[:2000,]
n_rep = 100
results_a = np.zeros(n_rep)
results_e = np.zeros(n_rep)
La = simulate_Cholesky_from_std(X, s2a=0.9, s2e=0.1, stability=1e-10)

for rep in range(n_rep):
    Z, y = simulate_from_raw_additive_remove_sampling_err(X, La, s2a=0.9, s2e=0.1)
    s2a_hat, s2e_hat, _ = MoM_additive(Z, y)
    results_a[rep] = s2a_hat
    results_e[rep] = s2e_hat


print("Summary:")
print(f"  s2a_hat: Mean = {results_a.mean():.3f}, Std = {results_a.std(ddof=1):.3f}")
print(f"  s2e_hat: Mean = {results_e.mean():.3f}, Std = {results_e.std(ddof=1):.3f}")

Summary:
  s2a_hat: Mean = 0.739, Std = 0.348
  s2e_hat: Mean = 0.263, Std = 0.344


In [181]:
import numpy as np


def simulate_contiguous_binomial(n, m, rho=0.95, mafs=None, seed=None):
    """
    Simulate genotypes with explicit Binomial(2, p_j) marginals and AR(1)-style
    LD using a two-state Markov chain on haplotype alleles.

    For each haplotype, allele at SNP j conditional on SNP j-1 follows:
        P(X_j = 1 | X_{j-1} = 1) = p_j + (1 - p_j) * rho_eff
        P(X_j = 1 | X_{j-1} = 0) = p_j * (1 - rho_eff)
    where rho_eff is set so the haplotype-level correlation between
    adjacent positions equals rho. With these probabilities, the marginal
    distribution at every SNP is Bernoulli(p_j) regardless of rho.

    Parameters
    ----------
    n : int
        Number of individuals.
    m : int
        Number of SNPs.
    rho : float in [0, 1)
        Haplotype-level correlation between adjacent SNPs.
        - rho = 0       → independent SNPs (HWE, no LD)
        - rho = 0.9     → moderate LD
        - rho = 0.99    → strong LD
        - rho = 0.999   → very strong, near-monolithic haplotype
    mafs : array of length m, or None
        Per-SNP MAFs. If None, drawn uniformly from [0.05, 0.5].
    seed : int or None
        Random seed.

    Returns
    -------
    X : (n, m) int8 array
        Genotype matrix with values in {0, 1, 2}.
    mafs : (m,) array
    """
    rng = np.random.default_rng(seed)
    if mafs is None:
        mafs = rng.uniform(0.05, 0.5, m)
    else:
        mafs = np.asarray(mafs, dtype=float)

    n_hap = 2 * n
    hap = np.empty((n_hap, m), dtype=np.int8)

    # First SNP: Bernoulli(p_0)
    hap[:, 0] = (rng.random(n_hap) < mafs[0]).astype(np.int8)

    # Subsequent SNPs: Markov transitions calibrated to give marginal p_j
    # at SNP j and correlation rho with SNP j-1.
    for j in range(1, m):
        p_prev = mafs[j - 1]
        p_curr = mafs[j]

        # Adjusted rho factor when MAFs differ across SNPs:
        # we use latent rho applied as a correlation on the haplotype indicators.
        # Conditional probabilities derived from
        #   Cov(X_{j-1}, X_j) = rho * sqrt(p_prev (1-p_prev) p_curr (1-p_curr))
        cov = rho * np.sqrt(p_prev * (1 - p_prev) * p_curr * (1 - p_curr))
        p_curr_given_prev1 = (p_prev * p_curr + cov) / p_prev
        p_curr_given_prev0 = (p_curr * (1 - p_prev) - cov) / (1 - p_prev)

        # Numerical safety: clip to [0, 1] (only matters at extreme MAF differences)
        p_curr_given_prev1 = np.clip(p_curr_given_prev1, 0.0, 1.0)
        p_curr_given_prev0 = np.clip(p_curr_given_prev0, 0.0, 1.0)

        prev = hap[:, j - 1]
        thresh = np.where(prev == 1, p_curr_given_prev1, p_curr_given_prev0)
        hap[:, j] = (rng.random(n_hap) < thresh).astype(np.int8)

    # Diploid: sum two haplotypes per individual
    X = hap[0::2] + hap[1::2]

    return X, mafs